## 1. Word embeddings with GloVe: install and load

GloVe is a ready-made embedding model. Each word has a vector of 50 numbers.
The first load downloads the model.

In [2]:
!python3 -m pip install gensim

In [3]:
import gensim.downloader as api

In [4]:
glove = api.load("glove-wiki-gigaword-50")

## 2. Vocabulary and word vectors

The model knows 400,000 words, and each one is a row in a table.
A word's vector is its row of numbers, here the first 6.

In [5]:
print("vocabulary size:", len(glove.key_to_index), "words   (whole words, no pieces)")
print("vector size    :", glove.vector_size)

vocabulary size: 400000 words   (whole words, no pieces)
vector size    : 50


In [6]:
print("word 'pod' is row :", glove.key_to_index["pod"])
print("its vector        :", glove["pod"][:6].round(2), "...")

word 'pod' is row : 22460
its vector        : [ 1.11 -0.07  0.8  -0.01  0.19  0.5 ] ...


## 3. Similarity with cosine

Cosine similarity shows how close two vectors are. Near 1 means similar meaning.
Related words (error, failure) score higher than unrelated ones (error, sandwich).

In [7]:
!python3 -m pip install numpy

In [8]:
import numpy as np

In [9]:
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

pairs = [("server", "machine"), ("server", "banana"),
         ("error", "failure"), ("error", "sandwich"),
         ("deploy", "release"), ("car", "truck"), ("car", "banana")]
for a, b in pairs:
    print(f"{a:8s} vs {b:9s} cosine = {cosine(glove[a], glove[b]):.2f}")

server   vs machine   cosine = 0.43
server   vs banana    cosine = 0.22
error    vs failure   cosine = 0.53
error    vs sandwich  cosine = 0.09
deploy   vs release   cosine = 0.30
car      vs truck     cosine = 0.92
car      vs banana    cosine = 0.22


## 4. Find the nearest words

most_similar returns the words closest in meaning.
The vectors group related words together.

In [10]:
for word in ["server", "linux", "outage", "deploy", "pod"]:
    print(f"{word:7s} ->", [w for w, _ in glove.most_similar(word, topn=5)])

server  -> ['servers', 'user', 'desktop', 'functionality', 'interface']
linux   -> ['unix', 'open-source', 'kernel', 'os', 'freebsd']
outage  -> ['outages', 'blackouts', 'glitch', 'blackout', 'shutdown']
deploy  -> ['deploying', 'deployed', 'deployment', 'troops', 'reinforcements']
pod     -> ['pods', 'larva', 'legume', 'larvae', 'worm']


## 5. Words missing from the vocabulary

GloVe knows only whole words, so new tech words such as kubernetes are missing.
This is why we use tokenizers with word pieces next.

In [11]:
for word in ["docker", "kubernetes", "crashloopbackoff"]:
    print(f"{word:17s} in GloVe vocabulary: {word in glove.key_to_index}")

docker            in GloVe vocabulary: True
kubernetes        in GloVe vocabulary: False
crashloopbackoff  in GloVe vocabulary: False


## 6. Transformer models: install and import

Hugging Face transformers gives us tokenizers and pre-trained models.
AutoTokenizer and AutoModel load them by name.

In [12]:
!python3 -m pip install transformers

In [13]:
from transformers import AutoTokenizer, AutoModel

/Users/prreddy/Documents/EDUeki/Courses/AIDevOps/github/AIDevOps/V1/embeddings/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 7. Fix the PyTorch version

These cells reinstall PyTorch at a fixed version (2.6.0).
Skip them if your environment already works.

In [14]:
!python3 -m pip uninstall -y torch torchvision torchaudio

Found existing installation: torch 2.6.0
Uninstalling torch-2.6.0:
  Successfully uninstalled torch-2.6.0
Found existing installation: torchvision 0.21.0
Uninstalling torchvision-0.21.0:
  Successfully uninstalled torchvision-0.21.0
Found existing installation: torchaudio 2.6.0
Uninstalling torchaudio-2.6.0:
  Successfully uninstalled torchaudio-2.6.0


In [15]:
!python3 -m pip install --no-cache-dir \
    --index-url https://pypi.org/simple \
    torch==2.6.0 \
    torchvision==0.21.0 \
    torchaudio==2.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 MB 19.6 MB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 17.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 17.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [torchaudio]3 [torchvision]


In [16]:
!python3 -m pip install --upgrade pip

## 8. Load the MiniLM model and tokenizer

all-MiniLM-L6-v2 is a small, fast sentence embedding model.
The tokenizer cuts text into tokens and the model turns them into vectors.

In [17]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"

In [18]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [19]:
model = AutoModel.from_pretrained(model_name)

Loading weights: 100%|█████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 25918.73it/s]


In [20]:
model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-5): 6 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
    

In [21]:
print("loaded:", model_name)

loaded: sentence-transformers/all-MiniLM-L6-v2


## 9. Tokenizer vocabulary and token IDs

The tokenizer has 30,522 tokens, and each token has an ID number.
Convert text to IDs, and IDs back to tokens.

In [22]:
vocab = tokenizer.get_vocab()          # a plain dictionary: token text -> token ID
print("vocabulary size:", len(vocab))

for word in ["pod", "server", "deploy", "error", "banana"]:
    print(f"{word:8s} -> token ID {vocab[word]}")

vocabulary size: 30522
pod      -> token ID 17491
server   -> token ID 8241
deploy   -> token ID 21296
error    -> token ID 7561
banana   -> token ID 15212


In [23]:
print(tokenizer.convert_ids_to_tokens([17491, 8241, 21296]))

['pod', 'server', 'deploy']


## 10. Tokenize a batch of sentences

input_ids are the token IDs. attention_mask marks real tokens (1) and padding (0).
Shorter sentences are padded so the batch is rectangular.

In [24]:
batch = tokenizer(["pod crashed after deploy", "node not ready"], padding=True, return_tensors="pt")
print("input_ids:\n", batch["input_ids"])
print("attention_mask:\n", batch["attention_mask"])
print("shape:", tuple(batch["input_ids"].shape), "= (sentences, tokens per sentence)")

input_ids:
 tensor([[  101, 17491,  8007,  2044, 21296,   102],
        [  101, 13045,  2025,  3201,   102,     0]])
attention_mask:
 tensor([[1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 0]])
shape: (2, 6) = (sentences, tokens per sentence)


## 11. The embedding table

The model has a table with one row per token and 384 numbers per row.
These numbers were learned when the model was trained.

In [28]:
embedding_matrix = model.get_input_embeddings().weight.detach()    # the table, learned during the model's training

rows, columns = embedding_matrix.shape
print("shape               :", tuple(embedding_matrix.shape))
print("rows (vocab size)   :", rows)
print("columns (vector size):", columns)
print("numbers in the table:", rows * columns)

shape               : (30522, 384)
rows (vocab size)   : 30522
columns (vector size): 384
numbers in the table: 11720448


## 12. From token IDs to vectors

Look up each token ID in the table to get its vector.
A sentence becomes one vector per token.

In [27]:
embedding_matrix = model.get_input_embeddings().weight.detach()    
sentence = "Pod crashed after deploy"
ids = tokenizer(sentence)["input_ids"]
tokens = tokenizer.convert_ids_to_tokens(ids)
vectors = embedding_matrix[ids]                # one row per token ID

print("shape:", tuple(vectors.shape), "= (tokens, vector size)\n")
for token, token_id, vector in zip(tokens, ids, vectors):
    print(f"{token:8s} ID {token_id:6d}  first 4 numbers: {vector[:4].numpy().round(3)}")

shape: (6, 384) = (tokens, vector size)

[CLS]    ID    101  first 4 numbers: [-0.018 -0.008  0.047 -0.087]
pod      ID  17491  first 4 numbers: [-0.007  0.017 -0.014 -0.017]
crashed  ID   8007  first 4 numbers: [ 0.022 -0.078 -0.003  0.037]
after    ID   2044  first 4 numbers: [ 0.     0.038 -0.052 -0.014]
deploy   ID  21296  first 4 numbers: [-0.051 -0.039  0.013  0.01 ]
[SEP]    ID    102  first 4 numbers: [ 0.033 -0.009 -0.04   0.01 ]
